# Phase 4: Validation & Rolling Backtest

This notebook runs validations on the optimized policies to verify their operational credibility. It performs Monte Carlo validation on future paths to capture performance distributions, executes a continuous rolling backtest over historical training data to evaluate dynamic parameter adaptation under real seasonality, and checks pass/fail diagnostic gates.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.config import Config
from src.data_loader import load_train_data, load_test_data, filter_families
from src.demand_uncertainty import calculate_ratios, center_ratios, cap_by_percentile
from src.distributions import bootstrap_sample
from src.simulation import InventorySimulator
from src.policies import SQPolicy, RSPolicy
from src.validation import ValidationEngine

## 1. Load Config and Data

In [2]:
config = Config('../configs/inventory_config.yaml')
train_df = load_train_data('../../Demand Forecasting System/data/actual_fitted_sales_on_train_final.csv', config)
test_df = load_test_data('../../Demand Forecasting System/data/actual_fitted_sales_on_test_final.csv', config)

train_df = filter_families(train_df, config.selected_families, config)
test_df = filter_families(test_df, config.selected_families, config)

## 2. Load Optimized Policies (Base Case)

In [3]:
# Load pre-optimized values from the pipeline output CSV if it exists, otherwise define from defaults
opt_results_path = '../outputs/policy_results/base_case_optimization_results.csv'
if os.path.exists(opt_results_path):
    opt_df = pd.read_csv(opt_results_path)
    print("Optimized policy parameters:")
    print(opt_df[['family', 'policy_type', 'safety_stock', 'reorder_point', 'order_quantity', 'order_up_to_level']])
else:
    print("Warning: Base-case CSV not found. Please run main.py or run optimizations first.")

Optimized policy parameters:
      family policy_type   safety_stock  reorder_point  order_quantity  \
0  GROCERY I       (s,Q)  487622.562927   1.750781e+06   314433.578236   
1  BEVERAGES       (s,Q)  588068.725480   1.565681e+06   260348.018483   
2   CLEANING       (R,S)       0.000000            NaN             NaN   

   order_up_to_level  
0                NaN  
1                NaN  
2      798371.817225  


## 3. Run Validation and Backtests

In [4]:
if os.path.exists(opt_results_path):
    # Seeded generator for paths
    n_reps = config.n_replications
    rng = np.random.default_rng(config.random_seed)
    
    col_family = config.columns['family']
    col_actual = config.columns['train_actual']
    col_fitted = config.columns['train_fitted']
    col_forecast = config.columns['test_forecast']
    
    for idx, row in opt_df.iterrows():
        family = row['family']
        p_type = config.policies.get(family, 'SQ')
        L = config.lead_times.get(family, 1)
        R = config.review_periods.get(family, 1)
        H_weekly = config.holding_cost_weekly
        K = config.order_cost
        B = config.shortage_costs.get(family, 15.0)
        
        hist_act = train_df[train_df[col_family] == family][col_actual].values
        hist_fit = train_df[train_df[col_family] == family][col_fitted].values
        test_f = test_df[test_df[col_family] == family][col_forecast].values
        
        # Generate paths
        n_weeks_test = len(test_f)
        ratios = calculate_ratios(hist_act, hist_fit, config.epsilon)
        ratios_centered = ratios / np.mean(ratios)
        ratios_capped = cap_by_percentile(ratios_centered, 1.0, 99.0)
        
        weekly_paths = np.zeros((n_reps, n_weeks_test))
        for i in range(n_reps):
            sampled_r = bootstrap_sample(ratios_capped, size=n_weeks_test, random_state=rng)
            weekly_paths[i, :] = np.maximum(0.0, test_f * sampled_r)
            
        # Setup Simulator
        simulator = InventorySimulator(lead_time=L, holding_cost=H_weekly, order_cost=K, shortage_cost=B)
        validator = ValidationEngine(simulator, target_fill_rate=config.target_fill_rate)
        
        # Define Policy Object
        if p_type.upper() == 'SQ':
            policy = SQPolicy(reorder_point=row['reorder_point'], order_quantity=row['order_quantity'])
            initial_inv = row['reorder_point']
        else:
            policy = RSPolicy(order_up_to_level=row['order_up_to_level'])
            initial_inv = row['order_up_to_level']
            
        # 3.1 Monte Carlo Validation
        val_res = validator.monte_carlo_validate(weekly_paths, policy, initial_inv)
        
        print(f"\n=== Validation: {family} ({p_type}) ===")
        print(f"  Monte Carlo Fill Rate: {val_res['fill_rate']:.4f} (Passed: {val_res['validation_passed']})")
        print(f"  Average Cost:          {val_res['total_cost']:.4f}")
        print(f"  Average Inventory:     {val_res['avg_inventory']:.2f}")
        
        # 3.2 Historical Rolling Backtest
        backtest_res = validator.rolling_backtest(
            historical_actuals=hist_act,
            historical_forecasts=hist_fit,
            policy_type=p_type,
            safety_stock=row['safety_stock'],
            order_quantity=row['order_quantity'] if p_type.upper() == 'SQ' else None
        )
        
        print(f"  Historical Backtest Realized Fill Rate: {backtest_res['fill_rate']:.4f}")
        print(f"  Historical Backtest Cost Index:         {backtest_res['total_cost']:.2f}")
        print(f"  Historical Backtest Average Inventory:  {backtest_res['avg_inventory']:.2f}")


=== Validation: GROCERY I (SQ) ===
  Monte Carlo Fill Rate: 0.9916 (Passed: True)
  Average Cost:          543439.8849
  Average Inventory:     609857.95
  Historical Backtest Realized Fill Rate: 0.9965
  Historical Backtest Cost Index:         17673542.27
  Historical Backtest Average Inventory:  637320.72

=== Validation: BEVERAGES (SQ) ===
  Monte Carlo Fill Rate: 0.9966 (Passed: True)
  Average Cost:          156552.7656
  Average Inventory:     687134.05
  Historical Backtest Realized Fill Rate: 1.0000
  Historical Backtest Cost Index:         7740.43
  Historical Backtest Average Inventory:  707348.10

=== Validation: CLEANING (RS) ===
  Monte Carlo Fill Rate: 1.0000 (Passed: True)
  Average Cost:          1361.6536
  Average Inventory:     446039.91
  Historical Backtest Realized Fill Rate: 1.0000
  Historical Backtest Cost Index:         6925.63
  Historical Backtest Average Inventory:  416219.59
